In [1]:
import sys
from groq import Groq
import json
import pydantic 
from pydantic import BaseModel
from typing import get_args, Literal
import os
from google import genai
from openai import OpenAI
from time import sleep

from prompts import *
from dataset_utils import load_or_download_dataset
from models import build_crf_model
from crf_utils import build_compact_schema, save_prediction, create_base_messages, invoke_model, reconstruct_orig_format_from_predictions
from api_keys import *

In [2]:
LANGUAGE = "it"

INPUT_MODEL = "gemini-3-flash-preview"
INPUT_FILE = f"{INPUT_MODEL}_{LANGUAGE}.jsonl"
INPUT_PATH = os.path.join("CRF-filling-CL4Health2026", "your_submission_data", INPUT_FILE)

# MODEL_ID = "qwen/qwen3-32b"
# MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"
MODEL_ID = "openai/gpt-oss-120b"
# MODEL_ID = "gemma-3-27b-it"
# MODEL_ID = "gemini-3-flash-preview" # "gemini-3-pro-review"
# MODEL_ID = "gpt-5-nano"
if "gemini" in MODEL_ID:
    LLM_TYPE = "gemini"
elif "gemma" in MODEL_ID:
    LLM_TYPE = "gemma"
elif "gpt" in MODEL_ID and "oss" not in MODEL_ID:
    LLM_TYPE = "openai"
else:
    LLM_TYPE = "groq"
OUTPUT_FILE = f"{MODEL_ID.split('/')[-1]}_re-elaborating-{INPUT_MODEL}_{LANGUAGE}.jsonl"
OUTPUT_PATH = os.path.join("CRF-filling-CL4Health2026", "your_submission_data", OUTPUT_FILE)

DATASET_TO_CRF = "NLP-FBK/dyspnea-crf-test"
MAX_RETRIES = 3

MODEL_ID, LLM_TYPE, INPUT_PATH, OUTPUT_PATH

('openai/gpt-oss-120b',
 'groq',
 'CRF-filling-CL4Health2026/your_submission_data/gemini-3-flash-preview_it.jsonl',
 'CRF-filling-CL4Health2026/your_submission_data/gpt-oss-120b_re-elaborating-gemini-3-flash-preview_it.jsonl')

In [3]:
os.chdir("..")

In [4]:
# one-time initializations
if LLM_TYPE=="groq":
    client = Groq(api_key=GROQ_API_KEY)
elif LLM_TYPE in ["gemma", "gemini"]:
    client = genai.Client(api_key=GOOGLE_API_KEY)
elif LLM_TYPE=="openai":
    client = OpenAI(api_key=OPENAI_API_KEY)
CRFModel = build_crf_model()

# Load development dataset
ds = load_or_download_dataset(DATASET_TO_CRF)[LANGUAGE]
print(f"{DATASET_TO_CRF} has {len(ds)} documents")

Found local cache at data/dyspnea-valid-options, loading from disk...
Dataset({
    features: ['options', 'item'],
    num_rows: 134
})
Found local cache at data/dyspnea-crf-test, loading from disk...
NLP-FBK/dyspnea-crf-test has 200 documents


In [5]:
id_to_predictions = dict()

with open(INPUT_PATH, "r") as f:
    for line in f:
        l = json.loads(line)
        id_to_predictions[l["document_id"]] = l["predictions"]

len(id_to_predictions)

200

In [6]:
# in case multiple runs are required for the same elaboraiton
processed_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
        processed_ids = {json.loads(line)["document_id"] for line in f}
len(processed_ids), processed_ids

(0, set())

In [7]:
from src.prompts import POST_ELABORATION_SYSTEM_PROMPT
#TODO clean this function
def crf_dataset():
    with open(OUTPUT_PATH, "a") as f:
        for item in ds:
            doc_id = item["document_id"]
            clinical_note = item["clinical_note"]
            predictions = reconstruct_orig_format_from_predictions(id_to_predictions[doc_id])

            if doc_id in processed_ids:
                print(f"Skipped {doc_id} because already processed")
                continue
            
            success = False
            
            print(f"Processing document: {doc_id}")

            for attempt in range(MAX_RETRIES):
                if "gemma" in MODEL_ID:
                    sleep(20)

                try:
                    # 1. Invoke model
                    output = invoke_model(
                        client,
                        MODEL_ID,
                        final_message=f"Clinical note: <<<{clinical_note}>>>\nJSON to verify: {predictions}",
                        messages=[],
                        attempt=attempt,
                        system_instruction=POST_ELABORATION_SYSTEM_PROMPT,
                        pydantic_model=CRFModel
                    )
                    
                    if isinstance(output, BaseModel):
                        result = output
                    else:
                        # 2. Parse JSON string
                        data_dict = json.loads(output)
                        # 3. Validate with Pydantic
                        result = CRFModel.model_validate(data_dict)
                    
                    # 4. Save
                    save_prediction(f, doc_id, result)
                    success = True
                    break # Exit retry loop on success
                except json.JSONDecodeError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] JSON Error for {doc_id}: {e}")
                    print(f"FAILED OUTPUT:\n{output}\n")
                    # Retry
                
                except pydantic.ValidationError as e:
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Validation Error for {doc_id}:")
                    print(e)
                    print(f"FAILED OUTPUT:\n{output}\n")
                    # Retry
                
                except Exception as e:
                    if "The model is overloaded" in str(e) or "currently over capacity" in str(e):
                        print(f"[Attempt {attempt+1}/{MAX_RETRIES}] Service Unavailable for {doc_id}: {e}")
                        attempt-=1
                        sleep(15)
                        continue
                    print(f"[Attempt {attempt+1}/{MAX_RETRIES}] System/API Error for {doc_id}: {e}")
                    return # TODO currently stopping in case the tokens for that day are over

            if not success:
                print(f"Skipping document {doc_id} after {MAX_RETRIES} failed attempts. Saving original output")
                
                # Fallback: restore previous results (nothing happened for this document)
                old_instance = CRFModel.model_validate(predictions)
                save_prediction(f, doc_id, old_instance)
        print("Done")


crf_dataset()

Processing document: 1003423_it
Processing document: 1051592_it
Processing document: 1160226_it
Processing document: 121138_it
Processing document: 1224596_it
Processing document: 1229118_it
Processing document: 1286408_it
Processing document: 1448461_it
Processing document: 1466535_it
Processing document: 1487935_it
Processing document: 1509420_it
Processing document: 1594629_it
Processing document: 1744582_it
Processing document: 1750279_it
Processing document: 1754455_it
Processing document: 199524_it
Processing document: 249048_it
Processing document: 340214_it
Processing document: 352393_it
Processing document: 354519_it
Processing document: 459381_it
Processing document: 508770_it
Processing document: 568647_it
Processing document: 577904_it
Processing document: 727813_it
Processing document: 776908_it
Processing document: 780560_it
Processing document: 782931_it
Processing document: 787171_it
Processing document: 81706_it
Processing document: 871508_it
Processing document: 90787

In [8]:
os.chdir("CRF-filling-CL4Health2026")
!python3 scoring.py --submission_path your_submission_data/{OUTPUT_FILE} --language {LANGUAGE}


=== Scoring program starting ===
Loading reference from development_data/dev_gt.jsonl
Loading submission from your_submission_data/gpt-oss-120b_re-elaborating-2nd-time-gemini-3-flash-preview_it.jsonl
TP: 276, FP: 372, FN: 78
Final macro-F1 = 0.6038067082955058
Scores written to your_sumbmission_scores/scores.json
=== Scoring program finished successfully ===

